In [ ]:
# ----------------------------------------------------
# 1. Setup and Installation
# ----------------------------------------------------
 
# Install core RAG libraries
!pip install -q sentence-transformers
!pip install -q langchain
!pip install -q langchain-experimental # Necessary for SemanticChunker
!pip install -q tqdm
!pip install -q google-colab
 
import os
import json
from google.colab import drive, files
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch
 
# Define Model Constants
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
EMBEDDING_DIM = 768
 
# Configuration based on user request (Chunk size is now advisory for the splitter)
# Semantic chunking focuses on MEANING, so chunk size is not strictly enforced.
# We will use the BGE model for the chunking process itself.
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100 # Overlap is not used in this specific semantic chunker
 
# Check for GPU and set the device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print("⚠️ WARNING: GPU not detected. Please go to Runtime -> Change runtime type and select 'T4 GPU'.")
print(f"✅ PyTorch device set to: {DEVICE}")
print(f"✅ Using BAAI Model: {EMBEDDING_MODEL_NAME} (Dimension: {EMBEDDING_DIM})")
print("✅ Chunking Strategy: Semantic Chunking (Splits based on topic shifts).")
 
# ----------------------------------------------------
# 2. Mount Drive and Load Scraped JSON
# ----------------------------------------------------
 
# 1. Mount Google Drive
drive.mount('/content/drive')
 
# 2. Interactive file selection
print("\nPlease select your web-scraped JSON file from your Google Drive path:")
uploaded = files.upload()
 
if not uploaded:
    print("❌ No file selected. Please run the cell again and select a file.")
    scraped_data = []
else:
    file_name = next(iter(uploaded))
    file_path = os.path.join(os.getcwd(), file_name)
 
    try:
        with open(file_path, 'r') as f:
            scraped_data = json.load(f)
 
        if not isinstance(scraped_data, list):
            scraped_data = [scraped_data]
 
        print(f"\n✅ Successfully loaded {len(scraped_data)} documents from {file_name}.")
    except Exception as e:
        print(f"\n❌ Error loading file: {e}")
        scraped_data = []
 
 
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
 
# ----------------------------------------------------
# 3. Semantic Chunking and Metadata Structuring
# ----------------------------------------------------
 
# --- 1. Define Wrapper Class (Same as before) ---
class BAAIEmbeddingsWrapper:
    """Wrapper to make SentenceTransformer work with LangChain's SemanticChunker."""
    def __init__(self, model_name, device):
        self.model = SentenceTransformer(model_name, device=device)
        self.device = device
 
    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_tensor=False).tolist()
 
    def embed_query(self, text):
        return self.model.encode([text], convert_to_tensor=False).tolist()[0]
 
# --- 2. Initialize Models (Same as before) ---
# Ensure you have defined EMBEDDING_MODEL_NAME and DEVICE variables previously
# e.g., EMBEDDING_MODEL_NAME = "BAAI/bge-m3", DEVICE = "cuda"
 
bge_embeddings = BAAIEmbeddingsWrapper(EMBEDDING_MODEL_NAME, DEVICE)
 
text_splitter = SemanticChunker(
    bge_embeddings,
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=95
)
 
milvus_entities = []
global_chunk_id = 0
 
print(f"\nStarting Semantic Chunking (using {EMBEDDING_MODEL_NAME})...")
 
# --- 3. Adjusted Loop for New JSON Format ---
 
for i, doc in enumerate(tqdm(scraped_data, desc="Semantic Chunking Documents")):
    content_to_chunk = doc.get("content", "")
 
    # Skip empty content
    if not content_to_chunk:
        continue
 
    # A. Semantic Split
    try:
        chunks = text_splitter.split_text(content_to_chunk)
    except Exception as e:
        print(f"Warning: Chunking failed for doc {i}. Error: {e}")
        continue
 
    # B. Extract Metadata (ADJUSTED)
    # Your JSON already has 'full_metadata_json' as a string.
    # We retrieve it directly. If missing, we default to an empty JSON string "{}".
    full_metadata_json_str = doc.get("full_metadata_json", "{}")
 
    # C. Create Entities
    for chunk in chunks:
        entity = {
            "id": f"sem-{global_chunk_id}",
            "chunk_text": chunk,
            "url": doc.get("url", "N/A"),
            "company": doc.get("company", "N/A"),
            "title": doc.get("title", "N/A"),
            "word_count": len(chunk.split()), # Word count for this specific chunk
            # Store the pre-formatted metadata string directly
            "full_metadata_json": full_metadata_json_str
        }
        milvus_entities.append(entity)
        global_chunk_id += 1
 
print(f"✅ Semantic Chunking complete. Total chunks created: {len(milvus_entities)}")
 
# ----------------------------------------------------
# 4. Generate Final BAAI Embeddings on GPU (Second Pass)
# ----------------------------------------------------
 
# The model is already loaded in Step 3, but we use the SentenceTransformer directly now
# for batch encoding to maximize GPU efficiency.
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
 
# Extract all text for batch processing
texts_to_embed = [entity["chunk_text"] for entity in milvus_entities]
 
print(f"Generating {EMBEDDING_DIM}-dimensional embeddings for {len(texts_to_embed)} semantic chunks using {DEVICE}...")
 
# Generate embeddings
embeddings = model.encode(
    texts_to_embed,
    show_progress_bar=True,
    batch_size=128,
    convert_to_tensor=True,
    device=DEVICE
).tolist()
 
# 5. Assemble the final data structure
final_milvus_data = []
for entity, vector in zip(milvus_entities, embeddings):
    final_entity = {
        "id": entity["id"],
        "content_vector": vector,
        "original_content": entity["chunk_text"],
        "url": entity["url"],
        "company": entity["company"],
        "title": entity["title"],
        "word_count": entity["word_count"],
        "full_metadata_json": entity["full_metadata_json"]
    }
    final_milvus_data.append(final_entity)
 
print(f"✅ Embedding generation complete. Final Vector Dimension: {len(final_milvus_data[0]['content_vector'])}")
print(f"Total records ready for Milvus insertion: {len(final_milvus_data)}")
 
# ----------------------------------------------------
# 5. Export and Download
# ----------------------------------------------------
 
OUTPUT_FILE_NAME = "milvus_ready_semantic_bge_base_data.json"
 
# Save the final data to a new JSON file
with open(OUTPUT_FILE_NAME, 'w') as f:
    json.dump(final_milvus_data, f, indent=2)
 
print(f"\n✅ Data successfully saved to {OUTPUT_FILE_NAME} in the Colab environment.")
 
# Trigger the download to your local machine
files.download(OUTPUT_FILE_NAME)
 
print("\n📦 Download of the final JSON file is complete.")
print("---")
print(f"NEXT STEP: Use this JSON file to insert data into your Milvus DB collection. The vector dimension required is **{EMBEDDING_DIM}**.")
 